In [ ]:
#!pip install astropy

In [ ]:
#pip install lightkurve

In [ ]:
import numpy as np
import matplotlib.pylab as plt
from astropy.io import fits
from sklearn.cluster import KMeans
import sys
import os
import lightkurve as lk
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("star_functions.py"), '..')))
import star_functions as nana

In [ ]:
fn = "/Users/natsuki/Projects/hogg_research/hoggnation/oscillator_catalog/good_parents_fit.fits"
with fits.open(fn) as hdu_list:
    print(hdu_list.info())
    data = hdu_list[1].data
    header = hdu_list[1].header
print(len(data), header)

In [ ]:
features = data["features"]
print(features.shape)

#create the time invariant feature vector for clustering (log(a0^2), log(a1^2 + b1^2), ...)
squared_feats = np.zeros((1438, 33))
for e,f in enumerate(features):
    j = 0
    for i in range(len(f) - 1):
        if i == 0:
            squared_feats[e][j] = f[i]**2
            j = j + 1
        if i%2 == 1:
            squared_feats[e][j] = f[i]**2 + f[i+1]**2
            j = j + 1


## plotting

In [ ]:
#from hogg
foo, k = squared_feats.shape
frequencies = np.outer(data["refined_frequency"], (1. + np.arange(k)))
informations = np.nansum(squared_feats * frequencies * (frequencies < 24.), axis=1)
print(informations)
refine_freqs = data["refined_frequency"]

In [ ]:
#from hogg
sizes = 0.5 * np.log10(informations)
sizes += 4.
sizes = np.clip(sizes, 0.01, None)
print(sizes)
for i, j in [(0, 1),
             (0, 2),
             (0, 3),
             (1, 2),(2,3),(1,3)]:
    plt.axhline(1.0, color="k", lw=1.0, alpha=0.5)
    plt.axvline(1.0, color="k", lw=1.0, alpha=0.5)
    plt.scatter(squared_feats[:, i], squared_feats[:, j], c=np.log10(refine_freqs), s=sizes)
    plt.loglog()
    plt.xlabel(f"scalar {i}")
    plt.ylabel(f"scalar {j}")
    plt.colorbar(label="log_10 frequency")
    plt.savefig(f"scatter_{i}_{j}.png")
    plt.show()

## running K-means

In [ ]:
#run with k = 3,10,30 with 3 random restarts, 9 plots total


#some stars with NaN values
good_mask = ~np.any(np.isnan(squared_feats), axis=1)

squared_feats = squared_feats[good_mask]
log_squared_feats = np.log10(squared_feats)
log_squared_feats = log_squared_feats[:, 1:] #get rid of constant?

randoms = np.array([42, 33,50])
syms = ["o", "x", "s", "D", "+", "*", "p", "^", "1"]
sizes = [3, 7, 11, 15, 18, 21]

### K = 3

In [ ]:
#final
K = 3

for r in randoms:
    kmeans = KMeans(n_clusters=K, init='k-means++', random_state=r)
    kmeans.fit(log_squared_feats)
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for ax, (i, j) in zip(axes, [(0, 1), (0, 2)]):
        for group in range(K):
            mask = labels == group
            sym = syms[group % len(syms)]
            size = sizes[group % len(sizes)]
            sc = ax.scatter(log_squared_feats[mask, i], log_squared_feats[mask, j],
                        c=[group] * mask.sum(),
                        vmin=0, vmax=2,
                        marker=sym,
                        s=size,
                        cmap='turbo',
                        linewidths=0.3,
                        edgecolors='black')
        ax.scatter(centroids[:, i], centroids[:, j], c='red', s=25, alpha=0.8, marker='o')
        xmin = log_squared_feats[:, 0].min()
        ymin = min(log_squared_feats[:, 1].min(), log_squared_feats[:, 2].min())

        # then inside the ax loop:
        ax.set_xlim(xmin, 0)
        ax.set_ylim(ymin, 0)
        ax.set_xlabel(f"log_10 scalar {i+1}")
        ax.set_ylabel(f"log_10 scalar {j+1}")
        ax.grid(True, alpha=0.3)
        fig.colorbar(sc, ax=ax, label="cluster")
    
    fig.suptitle(f"k = {K}, random restart {r}")
    plt.tight_layout()
    plt.show()

### phase folding for each group, K = 3

In [ ]:
K = 3

kmeans = KMeans(n_clusters=K, init='k-means++', random_state=r)
kmeans.fit(log_squared_feats)
labels = kmeans.labels_

for group in range(K):
    mask = labels == group
    phase_folding_at_freq(group,K,33)

### K = 10

In [ ]:
for r in randoms:
    kmeans = KMeans(n_clusters=10, init='k-means++', random_state=r)
    kmeans.fit(log_squared_feats)
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for ax, (i, j) in zip(axes, [(0, 1), (0, 2)]):
        for group in range(30):
            mask = labels == group
            sym = syms[group % len(syms)]
            size = sizes[group % len(sizes)]
            sc = ax.scatter(log_squared_feats[mask, i], log_squared_feats[mask, j],
                        c=[group] * mask.sum(),
                        vmin=0, vmax=9,
                        marker=sym,
                        s=size,
                        cmap='turbo',
                        linewidths=0.3,
                        edgecolors='black')
        ax.scatter(centroids[:, i], centroids[:, j], c='red', s=25, alpha=0.8, marker='o')
        xmin = log_squared_feats[:, 0].min()
        ymin = min(log_squared_feats[:, 1].min(), log_squared_feats[:, 2].min())

        # then inside the ax loop:
        ax.set_xlim(xmin, 0)
        ax.set_ylim(ymin, 0)
        ax.set_xlabel(f"log_10 scalar {i+1}")
        ax.set_ylabel(f"log_10 scalar {j+1}")
        ax.grid(True, alpha=0.3)
        fig.colorbar(sc, ax=ax, label="cluster")
    
    fig.suptitle(f"k = 10, random restart {r}")
    plt.tight_layout()
    plt.show()

### phase folding for each group, K = 10

In [ ]:
K = 10

kmeans = KMeans(n_clusters=K, init='k-means++', random_state=33)
kmeans.fit(log_squared_feats)
labels = kmeans.labels_

for group in range(K):
    phase_folding_at_freq(group, K, 33)

### k = 30

In [ ]:
#final
for r in randoms:
    kmeans = KMeans(n_clusters=30, init='k-means++', random_state=r)
    kmeans.fit(log_squared_feats)
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_
    print("centroids:", centroids[0])
    print(len(centroids))
    print("labels:", labels[0])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for ax, (i, j) in zip(axes, [(1, 0), (2, 0)]):
        for group in range(30):
            mask = labels == group
            sym = syms[group % len(syms)]
            size = sizes[group % len(sizes)]
            sc = ax.scatter(log_squared_feats[mask, i], log_squared_feats[mask, j],
                        c=[group] * mask.sum(),
                        vmin=0, vmax=29,
                        marker=sym,
                        s=size,
                        cmap='turbo',
                        linewidths=0.3,
                        edgecolors='black')
            
        ax.scatter(centroids[:, i], centroids[:, j], c='red', s=25, alpha=0.8, marker='o')
       
        ax.set_xlim(-11.9, 0)
        ax.set_ylim(-8.5, 0)
        ax.set_xlabel(f"log_10 scalar {i+1}")
        ax.set_ylabel(f"log_10 scalar {j+1}")
        ax.grid(True, alpha=0.3)
    
        fig.colorbar(sc, ax=ax, label="cluster")
    
    fig.suptitle(f"k = 30, random restart {r}")
    plt.show()

### phase folding for each group, K = 30

In [ ]:
K = 30

kmeans = KMeans(n_clusters=K, init='k-means++', random_state=33)
kmeans.fit(log_squared_feats)
labels = kmeans.labels_

for group in range(K):
    mask = labels == group
    phase_folding_at_freq(group, K, 33)

In [ ]:
print(len(log_squared_feats))
print(len(labels))

In [ ]:
#get rid of nan from features, refined frequencies, run this before funciton below
refined_freqs = data["refined_frequency"]
features = data["features"]
kics = np.array(data["star_id"])
good_mask = ~np.any(np.isnan(features), axis=1)
features = features[good_mask]
refined_freqs = refined_freqs[good_mask]
kics = kics[good_mask]
print(len(kics))
print(kics[:10])
print(len(log_squared_feats))

In [ ]:
def phase_folding_at_freq(group, k, random_restart):
    mask = labels == group
    true_indices = np.where(mask)[0] #get indices where its true in mask
    #i = true_indices[2]#choose the first star indec
    dists = np.linalg.norm(log_squared_feats[true_indices] - centroids[group], axis=1)
    i = true_indices[np.argmin(dists)] #takes index from label array to find corresponding 1400 index
    print(i)
    star_id, feature, freq = kics[i], features[i], refined_freqs[i]

    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)
    
    theta = np.linspace(0, 4*np.pi, 1000)
    yplot0 = np.zeros_like(theta)

    for m in range(1, 33):
        a = feature[2*m-1]
        b = feature[2*m]
        yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)
    #yplot0 += feature[0]
        
    omega = 2 * np.pi * freq
    phase = (omega * t_fit) % (4 * np.pi)
    yplot = flux_fit - np.nanmean(flux_fit)

    plt.plot(phase, yplot, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k')
    plt.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
    plt.xlabel('Phase')
    plt.ylabel('Flux')
    plt.title(f'phase fold @ freq, K = {k}, group = {group}, random_restart {random_restart}')
    plt.show()
    


In [ ]:
phase_folding_at_freq(0, 10, 33)